In [ ]:
# Example 1: top-level public API is exposed via __all__.
import python_discovery

expected = {
    "KNOWN_ARCHITECTURES",
    "KNOWN_IMPLEMENTATIONS",
    "ContentStore",
    "DiskCache",
    "PyInfoCache",
    "PythonInfo",
    "PythonSpec",
    "SimpleSpecifier",
    "SimpleSpecifierSet",
    "SimpleVersion",
    "get_interpreter",
    "iter_interpreters",
    "normalize_isa",
}
missing = [name for name in expected if not hasattr(python_discovery, name)]
print(f"missing public names: {missing}")
assert not missing, f"missing public names: {missing}"

In [ ]:
# Example 2: parse interpreter specifications via PythonSpec.from_string_spec.
from python_discovery import PythonSpec

samples = ["python3.12", "3.11", "cpython3.10", ">=3.11,<3.13"]
parsed = []
for s in samples:
    spec = PythonSpec.from_string_spec(s)
    parsed.append((s, repr(spec)))
    print(f"{s!r:30s} -> {spec!r}")

assert len(parsed) == len(samples)
# Each spec should be a PythonSpec instance.
assert all(isinstance(PythonSpec.from_string_spec(s), PythonSpec) for s in samples)

In [ ]:
# Example 3: SimpleVersion + SimpleSpecifier + SimpleSpecifierSet semantics.
from python_discovery import SimpleSpecifier, SimpleSpecifierSet, SimpleVersion

v_a = SimpleVersion.from_string("3.11.7")
v_b = SimpleVersion.from_string("3.12.0")
print(f"v_a={v_a!r}  v_b={v_b!r}")
assert v_a < v_b

spec = SimpleSpecifier.from_string(">=3.10")
print(f"spec={spec!r}  v_a in spec? {spec.contains(v_a)}")
assert spec.contains(v_a)

spec_set = SimpleSpecifierSet.from_string(">=3.10,<3.12")
print(f"spec_set={spec_set!r}  v_a in set? {spec_set.contains(v_a)}  v_b in set? {spec_set.contains(v_b)}")
assert spec_set.contains(v_a)
assert not spec_set.contains(v_b)

In [ ]:
# Example 4: discover the currently-running interpreter via get_interpreter + DiskCache.
import sys
import tempfile
from pathlib import Path

from python_discovery import DiskCache, PythonInfo, get_interpreter

cur = f"{sys.version_info.major}.{sys.version_info.minor}"
with tempfile.TemporaryDirectory() as td:
    cache = DiskCache(root=Path(td))
    result = get_interpreter(cur, cache=cache)
    print(f"requested spec: {cur}")
    print(f"result type:    {type(result).__name__}")
    if result is not None:
        print(f"executable:     {result.executable}")
        print(f"implementation: {result.implementation}")
        print(f"version_info:   {result.version_info[:3]}")
        assert isinstance(result, PythonInfo)
        assert result.version_info[0] == sys.version_info.major
        assert result.version_info[1] == sys.version_info.minor

In [ ]:
# Example 5: iter_interpreters yields PythonInfo objects + KNOWN_* constants are non-empty.
import tempfile
from pathlib import Path

from python_discovery import KNOWN_ARCHITECTURES, KNOWN_IMPLEMENTATIONS, DiskCache, PythonInfo, iter_interpreters, normalize_isa

print(f"KNOWN_IMPLEMENTATIONS ({len(KNOWN_IMPLEMENTATIONS)}): {sorted(KNOWN_IMPLEMENTATIONS)[:5]}")
print(f"KNOWN_ARCHITECTURES   ({len(KNOWN_ARCHITECTURES)}): {sorted(KNOWN_ARCHITECTURES)[:5]}")
print(f"normalize_isa('x86_64') = {normalize_isa('x86_64')!r}")
assert KNOWN_IMPLEMENTATIONS and KNOWN_ARCHITECTURES

with tempfile.TemporaryDirectory() as td:
    cache = DiskCache(root=Path(td))
    found = []
    for info in iter_interpreters(">=3.8", cache=cache):
        found.append(info)
        if len(found) >= 3:
            break
    for info in found:
        assert isinstance(info, PythonInfo)
        print(f"  {info.executable}  v={info.version_info[:3]}")

assert True, "python_discovery smoke test complete"